# Trajectory and velocity analysis of scRNAseq COLON data 

This notebook will guide your through the analysis... This is an example for the INFLAMED dataset. Other datasets (HEALTHY, PROGEN etc.) are analysed accordingly.

## 0. Imports and settings

In [ ]:
import scanpy as sc
import scvelo as scv
import numpy as np
import pandas as pd
#import mnnpy
import matplotlib.pyplot as plt
import seaborn as sb
#import scrublet as scr
#import doubletdetection
import scanpy.api as scapi

# Box for text background in PAGA
bbox = dict(bbox=dict(boxstyle="round", ec='white', fc="white", alpha=0.5, linewidth=0))

# List of genes associated with cell cycle.
cell_cycle_genes = [x.strip() for x in open('data/cell_cycle_genes.txt')]
cell_cycle_genes = cell_cycle_genes[1:]
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]

# Proliferation markers
ProlifMarkers = ['RAD51C','SLC7A2','CCDC18','DCTD','RAD54B','BARD1','KLHL23','FIGNL1','POLA1','DEPDC1','PPIL5','PPAT','C6ORF150','XRCC2','SKA1','SLC12A2','RCC2','KIF18A','KIF11','ESCO2','E2F7','RAD54L','CHEK1','PRKD3','NA','CLSPN','KIAA0101','DTL','SLC7A2','ATAD5','POLE','FANCB','CENPA','NEXN','PPIL5','FAM111A','TTPA','CDC7','NAP1L1','HEMGN','KNTC1','PRKD3','TBC1D19','SKA3','NCAPG2','POLE2','EXO1','CENPI','SGOL2','CENPN','DTL','CENPN','NEIL3','EXO1','HMMR','RAD54L','BUB1','MCM3','NRM','CNN3','ALMS1','TIMELESS','ATAD2','RRP1B','AURKB','SLC16A12','RAD51C','MCM3','TPX2','C1ORF135','KIFC1','TBC1D4','CHEK1','ZWILCH','SCML2','FADS1','GINS2','MYC','TIMELESS','AQP4','PCK2','CDC45L','ANKRD26','PPAT','HELLS','C15ORF42','ZNF473','CDCA2','NA','HMMR','UBE2T','BRCA1','KIF11','KIF20B','PLK4','PRR11','TMEFF1','CNN3','NA','MCM5','MCM2','GINS1','TMEM107','ERCC6L','CASC5','IFITM3','RAD51AP1','RAD51AP1','BUB1','NCAPH','CBX6','DTL','BLM','GINS2','PRR11','9430037O13RIK','MORC4','PFAS','FGR','RAD54B','C6ORF173','NCAPH','NA','PALB2','MELK','NSL1','CDCA2','NAP1L1','C6ORF150','MBD4','MCM3','CENPF','MCM7','RACGAP1','TIPIN','MYB','C14ORF106','C16ORF48','RAD18','CENPE','RAD51AP1','CHEK1','MCM3','UHRF1','CEP55','FOXM1','SLFN13','IQGAP3','TUBE1','NUF2','CCDC99','RAD51','FAM72A','CDCA5','SMC2','KIF18B','MCM3','RACGAP1','GLS2','DUT','TMEM107','BRCA1','AU016916','MYB','2700099C18RIK','GTSE1','RAD54L','HELLS','WDR35','PLK1','TIMP3','RPA2','CEP55','NA','RAD18','C6ORF167','C4ORF21','PRIM2','C12ORF48','BUB1B','C1ORF112','NA','LAMC1','ANLN','TRIM37','RRM2','CCNA2','HAUS5','ESPL1','TCF19','FOXM1','DDX11','KIF2C','CCNE1','PUS7','LPHN1','KBTBD6','NA','TXNDC16','DACH1','MASTL','TRIM37','ZRANB3','SIVA1','2810454L23RIK','CELSR2','PSIP1','MCM8','SHCBP1','ASF1B','KCNN4','AU020206','PRIM1','CKS1B','CHAF1B','MCM2','RCC2','RASA3','DUT','CTPS','BAG2','SIVA1','CKLF','PRIM2','DUS4L','TMEM194B','TUBB','HAUS6','DCAF12L1','MARCKS','SETDB1','2810408B13RIK','SYCE2','UNG','NA','SEMA3C','CLCA4','MLF1IP','PAICS','MCM7','POLD1','C17ORF53','E2F1','ZC3H7B','TRAIP','LOC388559','MPHOSPH9','WDHD1','BCL7A','NA','ATIC','CCBE1','TSGA14','HELLS','MASTL','SPAG5','LYSMD2','C15ORF23','CEP76','DNAJC18','TAF4B','CDC6','SMC2','CENPE','4930513N10RIK','LYAR','CDK2','FOXM1','CKLF','NCAPD2','SLFN13','RPUSD2','VRK1','CYP39A1','DLGAP5','INCENP','PBK','TMEFF1','NUSAP1','NEK2','C3ORF26','VRK1','PSMC3IP','POLI','NFATC2','D17H6S56E-5','PGM2L1','AURKB','CEP97','CKAP2L','FANCB','DNMT1','CCNF','BRCA1','ASPM','HMMR','HMGA2','DSN1','NA','CDCA5','PGM2L1','IGF1R','ZNF367','WDR34','NUP210','EXOSC8','GEN1','LIG1','TPX2','HMMR','TK1','PLK4','TRIM68','HAUS6','FAM54A','C8ORF79','AEN','ZNRF3','PSMC3IP','CCDC14','DNA2','CENPP','C21ORF91','DLGAP5','DCK','SLC7A5','HAUS4','CHTF18','SEMA4D','PGM2L1','ZBTB25','UHRF1','TSGA14','BCKDHB','MCM4','KIF22','CHEK1','NAP1L1','RRM1','FTSJD1','SGOL1','MDN1','FEN1','RRM2','CKLF','NEK2','QSOX2','RRP15','NCAPG','AURKA','MYBL2','NA','PAICS','PIP4K2B','SPC24','DNA2','PRC1','RPP40','KIF24','MPHOSPH9','TMEM173','TMEM48','NIN','WDR76','ZNF275','UTP15','3110040M04RIK','MTHFD2','TACC3','NA','RBBP6','CENPF','ZMYND19','SKA2','MTBP','NDE1','MYBL2','EXOSC2','DIAPH3','CDC2','CP110','KIAA0649','GEMIN5','DUSP7','POLR1E','NUP133','TRIM37','TSPAN12','NAP1L1','PRC1','GEMIN4','GINS3','TOPBP1','NEUROG3','MPHOSPH9','CASP12','WWTR1','NAP1L1','MCM7','RFC3','ZMYND19','KIF4A','LMNB1','RFC3','BBS7','ILF3','ZNF704','GSTCD','PRIM1','NA','ZBTB16','NA','DKC1','QTRT1','TRIP13','SLC1A5','CDCA7L','C14ORF143','MCM5','TRIP13','2810442I21RIK','DNAH11','4933439C10RIK','POLD1','CKLF','CEP192','MIRHG1','ZFP783','NUP85','C3ORF26','TLR2','C6ORF167']
ProlifMarkers = list(set(ProlifMarkers))

## 1. Load data

In [ ]:
sample_name = "inflamed3"
file = "data/{}.loom".format(sample_name)
adata3 = scv.read_loom(file, sparse=True, cleanup=True)
adata3.var_names_make_unique()
# Delete unnecessary data
del adata3.obs['Clusters']
del adata3.obs['_X']
del adata3.obs['_Y']
del adata3.var['Accession']
del adata3.var['Chromosome']
del adata3.var['End']
del adata3.var['Start']
del adata3.var['Strand']
del adata3.layers['ambiguous']

sample_name = "inflamed4"
file = "data/{}.loom".format(sample_name)
adata4 = scv.read_loom(file, sparse=True, cleanup=True)
adata4.var_names_make_unique()
# Delete unnecessary data
del adata4.obs['Clusters']
del adata4.obs['_X']
del adata4.obs['_Y']
del adata4.var['Accession']
del adata4.var['Chromosome']
del adata4.var['End']
del adata4.var['Start']
del adata4.var['Strand']
del adata4.layers['ambiguous']

sample_name = "inflamed6"
file = "data/{}.loom".format(sample_name)
adata6 = scv.read_loom(file, sparse=True, cleanup=True)
adata6.var_names_make_unique()
# Delete unnecessary data
del adata6.obs['Clusters']
del adata6.obs['_X']
del adata6.obs['_Y']
del adata6.var['Accession']
del adata6.var['Chromosome']
del adata6.var['End']
del adata6.var['Start']
del adata6.var['Strand']
del adata6.layers['ambiguous']

sample_name = "inflamed10"
file = "data/{}.loom".format(sample_name)
adata10 = scv.read_loom(file, sparse=True, cleanup=True)
adata10.var_names_make_unique()
# Delete unnecessary data
del adata10.obs['Clusters']
del adata10.obs['_X']
del adata10.obs['_Y']
del adata10.var['Accession']
del adata10.var['Chromosome']
del adata10.var['End']
del adata10.var['Start']
del adata10.var['Strand']
del adata10.layers['ambiguous']

## 2. Quality control

In [ ]:
# Concatenate the files to calculate joint QC metrics
adata_qc = adata3.concatenate(adata4, adata6, adata10,
                              batch_key='Sample', 
                              batch_categories=['inflamed3', 'inflamed4', 'inflamed6', 'inflamed10'])

# Calculate QC covariates
adata_qc.obs['n_counts'] = adata_qc.X.sum(1)
adata_qc.obs['n_genes'] = (adata_qc.X > 0).sum(1)
adata_qc.obs['n_spliced'] = adata_qc.layers['spliced'].sum(1)
adata_qc.obs['n_unspliced'] = adata_qc.layers['unspliced'].sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata_qc.var_names.str.startswith('MT-')
adata_qc.obs['percent_mito'] = np.sum(
    adata_qc[:, mito_genes].X, axis=1) / np.sum(adata_qc.X, axis=1)
ribo_genes = adata_qc.var_names.str.startswith('RP')
adata_qc.obs['percent_ribo'] = np.sum(
    adata_qc[:, ribo_genes].X, axis=1) / np.sum(adata_qc.X, axis=1)
adata_qc.obs['percent_MALAT1'] = np.sum(
    adata_qc[:, 'MALAT1'].X, axis=1) / np.sum(adata_qc.X, axis=1)

# Initial plotting settings for more detailed scatter and dist plots.
scv.settings.set_figure_params('scvelo', dpi=150, vector_friendly=False)

sc.pl.violin(adata_qc, ['n_genes', 'n_counts'], jitter=0.4, groupby='Sample')
sc.pl.violin(adata_qc, ['percent_mito', 'percent_ribo'], jitter=0.4, groupby='Sample')
sc.pl.violin(adata_qc, ['n_spliced', 'n_unspliced'], jitter=0.4, groupby='Sample')

### 2.1 inflamed3 QC 

In [ ]:
# Calculate QC covariates
adata3.obs['n_counts'] = adata3.X.sum(1)
adata3.obs['log_counts'] = np.log(adata3.obs['n_counts'])
adata3.obs['n_spliced'] = adata3.layers['spliced'].sum(1)
adata3.obs['n_unspliced'] = adata3.layers['unspliced'].sum(1)
adata3.obs['n_genes'] = (adata3.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata3.var_names.str.startswith('MT-')
adata3.obs['percent_mito'] = np.sum(
    adata3[:, mito_genes].X, axis=1) / np.sum(adata3.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata3.var_names.str.startswith('RP')
adata3.obs['percent_ribo'] = np.sum(
    adata3[:, ribo_genes].X, axis=1) / np.sum(adata3.X, axis=1)

adata3.obs['percent_MALAT1'] = np.sum(
    adata3[:, 'MALAT1'].X, axis=1) / np.sum(adata3.X, axis=1)

sc.pl.scatter(adata3, 'n_counts', 'n_genes', color='percent_mito', title='inflamed3 (percent_mito)')
sc.pl.scatter(adata3, 'n_spliced', 'n_unspliced')

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata3.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
scrub.plot_histogram()

In [ ]:
scrub.set_embedding('UMAP', scr.get_umap(scrub.manifold_obs_, 10, min_dist=0.3))
scrub.plot_embedding('UMAP', order_points=True)

In [ ]:
adata3.obs['doublet_score'] = scrub.doublet_scores_obs_

Another tool to annotate doublets is Doublet Detection. More sophisticated, we use it to actually call doublets.

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata3.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f = doubletdetection.plot.convergence(clf, show=True, p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f2, umap_coords = doubletdetection.plot.umap_plot(adata3.X, doublets, random_state=1, show=True)

In [ ]:
adata3.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
#Thresholding decision: spliced counts
ax=sb.distplot(adata3.obs['n_spliced'], kde=False)
plt.show()

ax=sb.distplot(adata3.obs['n_spliced'][adata3.obs['n_spliced']<5000], kde=False, bins=80)
plt.show()

ax=sb.distplot(adata3.obs['n_spliced'][adata3.obs['n_spliced']>17500], kde=False, bins=80)
plt.show()

In [ ]:
#Thresholding decision: unspliced counts
ax=sb.distplot(adata3.obs['n_unspliced'], kde=False)
plt.show()

ax=sb.distplot(adata3.obs['n_unspliced'][adata3.obs['n_unspliced']<2000], kde=False, bins=60)
plt.show()

In [ ]:
#Thresholding decision: genes
ax=sb.distplot(adata3.obs['n_genes'], kde=False, bins=80)
plt.show()
ax=sb.distplot(adata3.obs['n_genes'][adata3.obs['n_genes']<2000], kde=False, bins=80)
plt.show()

In [ ]:
print("Before filtering cells:\n", adata3, "\n")

sc.pp.filter_cells(adata3, min_counts=2000)
sc.pp.filter_cells(adata3, max_counts=23500)
adata3 = adata3[adata3.obs['n_unspliced']>750]
sc.pp.filter_cells(adata3, min_genes=750)

print("After filtering cells:\n", adata3)

Let's now look at the mitochondrial content.

In [ ]:
sc.pl.violin(adata3, ['n_genes', 'n_counts', 'percent_mito'],
             jitter=0.4, multi_panel=True)
sc.pl.scatter(adata3, 'n_genes', 'percent_mito')

In [ ]:
adata3 = adata3[adata3.obs['percent_mito'] < 0.3, :]
sc.pl.scatter(adata3, 'n_counts', 'n_genes', color='percent_mito', title='inflamed3 (percent_mito)')
print(adata3)

In [ ]:
sc.pl.scatter(adata3, 'n_genes', 'doublet_score', color='doublet', palette=['gray', 'orange'], legend_loc='none', title='Doublet')
adata3 = adata3[adata3.obs['doublet'] != '1']
adata3 = adata3[adata3.obs['doublet_score'] < 0.3]

In [ ]:
sc.pl.highest_expr_genes(adata3, n_top=20)
print(adata3)

### 2.2 inflamed4 QC

In [ ]:
# Calculate QC covariates
adata4.obs['n_counts'] = adata4.X.sum(1)
adata4.obs['log_counts'] = np.log(adata4.obs['n_counts'])
adata4.obs['n_spliced'] = adata4.layers['spliced'].sum(1)
adata4.obs['n_unspliced'] = adata4.layers['unspliced'].sum(1)
adata4.obs['n_genes'] = (adata4.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata4.var_names.str.startswith('MT-')
adata4.obs['percent_mito'] = np.sum(
    adata4[:, mito_genes].X, axis=1) / np.sum(adata4.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata4.var_names.str.startswith('RP')
adata4.obs['percent_ribo'] = np.sum(
    adata4[:, ribo_genes].X, axis=1) / np.sum(adata4.X, axis=1)

adata4.obs['percent_MALAT1'] = np.sum(
    adata4[:, 'MALAT1'].X, axis=1) / np.sum(adata4.X, axis=1)

sc.pl.scatter(adata4, 'n_counts', 'n_genes', color='percent_mito', title='inflamed4 (percent_mito)')
sc.pl.scatter(adata4, 'n_spliced', 'n_unspliced')

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata4.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
scrub.plot_histogram()

In [ ]:
scrub.set_embedding('UMAP', scr.get_umap(scrub.manifold_obs_, 10, min_dist=0.3))
scrub.plot_embedding('UMAP', order_points=True)

In [ ]:
adata4.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata4.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f = doubletdetection.plot.convergence(clf, show=True, p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f2, umap_coords = doubletdetection.plot.umap_plot(adata4.X, doublets, random_state=1, show=True)

In [ ]:
adata4.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
#Thresholding decision: spliced counts
ax=sb.distplot(adata4.obs['n_spliced'], kde=False)
plt.show()

ax=sb.distplot(adata4.obs['n_spliced'][adata4.obs['n_counts']<2500], kde=False, bins=60)
plt.show()

ax=sb.distplot(adata4.obs['n_spliced'][adata4.obs['n_counts']>20000], kde=False, bins=60)
plt.show()

In [ ]:
#Thresholding decision: unspliced counts
ax=sb.distplot(adata4.obs['n_unspliced'], kde=False)
plt.show()

ax=sb.distplot(adata4.obs['n_unspliced'][adata4.obs['n_unspliced']<2000], kde=False, bins=60)
plt.show()

In [ ]:
#Thresholding decision: genes
ax=sb.distplot(adata4.obs['n_genes'], kde=False, bins=60)
plt.show()
ax=sb.distplot(adata4.obs['n_genes'][adata4.obs['n_genes']<1000], kde=False, bins=60)
plt.show()

In [ ]:
print("Before filtering cells:\n", adata4, "\n")

sc.pp.filter_cells(adata4, min_counts=1600)
sc.pp.filter_cells(adata4, max_counts=27500)
adata4 = adata4[adata4.obs['n_unspliced']>900]
sc.pp.filter_cells(adata4, min_genes=500)

print("After filtering cells:\n", adata4)

In [ ]:
sc.pl.violin(adata4, ['n_genes', 'n_counts', 'percent_mito'],
             jitter=0.4, multi_panel=True)
sc.pl.scatter(adata4, 'n_genes', 'percent_mito')

In [ ]:
adata4 = adata4[adata4.obs['percent_mito'] < 0.3, :]
sc.pl.scatter(adata4, 'n_counts', 'n_genes', color='percent_mito', title='inflamed4 (percent_mito)')
print(adata4)

In [ ]:
sc.pl.scatter(adata4, 'n_genes', 'doublet_score', color='doublet', palette=['gray', 'orange'], legend_loc='none', title='Doublet')
adata4 = adata4[adata4.obs['doublet'] != '1']
adata4 = adata4[adata4.obs['doublet_score'] < 0.3]

In [ ]:
sc.pl.highest_expr_genes(adata4, n_top=20)
print(adata4)

### 2.3 inflamed6 QC

In [ ]:
# Calculate QC covariates
adata6.obs['n_counts'] = adata6.X.sum(1)
adata6.obs['log_counts'] = np.log(adata6.obs['n_counts'])
adata6.obs['n_spliced'] = adata6.layers['spliced'].sum(1)
adata6.obs['n_unspliced'] = adata6.layers['unspliced'].sum(1)
adata6.obs['n_genes'] = (adata6.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata6.var_names.str.startswith('MT-')
adata6.obs['percent_mito'] = np.sum(
    adata6[:, mito_genes].X, axis=1) / np.sum(adata6.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata6.var_names.str.startswith('RP')
adata6.obs['percent_ribo'] = np.sum(
    adata6[:, ribo_genes].X, axis=1) / np.sum(adata6.X, axis=1)

adata6.obs['percent_MALAT1'] = np.sum(
    adata6[:, 'MALAT1'].X, axis=1) / np.sum(adata6.X, axis=1)

sc.pl.scatter(adata6, 'n_counts', 'n_genes', color='percent_mito', title='inflamed6 (percent_mito)')
sc.pl.scatter(adata6, 'n_spliced', 'n_unspliced')

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata6.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
scrub.plot_histogram()

In [ ]:
scrub.set_embedding('UMAP', scr.get_umap(scrub.manifold_obs_, 10, min_dist=0.3))
scrub.plot_embedding('UMAP', order_points=True)

In [ ]:
adata6.obs['doublet_score'] = scrub.doublet_scores_obs_

Another tool to annotate doublets is Doublet Detection. More sophisticated, we use it to actually call doublets.

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata6.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f = doubletdetection.plot.convergence(clf, show=True, p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f2, umap_coords = doubletdetection.plot.umap_plot(adata6.X, doublets, random_state=1, show=True)

In [ ]:
adata6.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
#Thresholding decision: spliced counts
ax=sb.distplot(adata6.obs['n_spliced'], kde=False)
plt.show()

ax=sb.distplot(adata6.obs['n_spliced'][adata6.obs['n_counts']<6000], kde=False, bins=60)
plt.show()

ax=sb.distplot(adata6.obs['n_spliced'][adata6.obs['n_counts']>10000], kde=False, bins=60)
plt.show()

In [ ]:
#Thresholding decision: unspliced counts
ax=sb.distplot(adata6.obs['n_unspliced'], kde=False)
plt.show()

ax=sb.distplot(adata6.obs['n_unspliced'][adata6.obs['n_unspliced']<2000], kde=False, bins=60)
plt.show()

In [ ]:
#Thresholding decision: genes
ax=sb.distplot(adata6.obs['n_genes'], kde=False, bins=60)
plt.show()
ax=sb.distplot(adata6.obs['n_genes'][adata6.obs['n_genes']<1250], kde=False, bins=60)
plt.show()

In [ ]:
print("Before filtering cells:\n", adata6, "\n")

sc.pp.filter_cells(adata6, min_counts=3100)
sc.pp.filter_cells(adata6, max_counts=14000)
adata6 = adata6[adata6.obs['n_unspliced']>900]
sc.pp.filter_cells(adata6, min_genes=900)

print("After filtering cells:\n", adata6)

In [ ]:
sc.pl.violin(adata6, ['n_genes', 'n_counts', 'percent_mito'],
             jitter=0.4, multi_panel=True)
sc.pl.scatter(adata6, 'n_genes', 'percent_mito')

In [ ]:
adata6 = adata6[adata6.obs['percent_mito'] < 0.4, :]
sc.pl.scatter(adata6, 'n_counts', 'n_genes', color='percent_mito', title='inflamed6 (percent_mito)')
print(adata6)

In [ ]:
sc.pl.scatter(adata6, 'n_genes', 'doublet_score', color='doublet', palette=['gray', 'orange'], legend_loc='none', title='Doublet')
adata6 = adata6[adata6.obs['doublet'] != '1']
adata6 = adata6[adata6.obs['doublet_score'] < 0.3]

In [ ]:
sc.pl.highest_expr_genes(adata6, n_top=20)
print(adata6)

### 2.4 inflamed10 QC

The analysis and its reasoning is analogous to the previous one. See section **2.1** for describtion.

In [ ]:
# Calculate QC covariates
adata10.obs['n_counts'] = adata10.X.sum(1)
adata10.obs['log_counts'] = np.log(adata10.obs['n_counts'])
adata10.obs['n_spliced'] = adata10.layers['spliced'].sum(1)
adata10.obs['n_unspliced'] = adata10.layers['unspliced'].sum(1)
adata10.obs['n_genes'] = (adata10.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata10.var_names.str.startswith('MT-')
adata10.obs['percent_mito'] = np.sum(
    adata10[:, mito_genes].X, axis=1) / np.sum(adata10.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata10.var_names.str.startswith('RP')
adata10.obs['percent_ribo'] = np.sum(
    adata10[:, ribo_genes].X, axis=1) / np.sum(adata10.X, axis=1)

adata10.obs['percent_MALAT1'] = np.sum(
    adata10[:, 'MALAT1'].X, axis=1) / np.sum(adata10.X, axis=1)

sc.pl.scatter(adata10, 'n_counts', 'n_genes', color='percent_mito', title='inflamed10 (percent_mito)')
sc.pl.scatter(adata10, 'n_spliced', 'n_unspliced')

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata10.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
scrub.plot_histogram()

In [ ]:
scrub.set_embedding('UMAP', scr.get_umap(scrub.manifold_obs_, 10, min_dist=0.3))
scrub.plot_embedding('UMAP', order_points=True)

In [ ]:
adata10.obs['doublet_score'] = scrub.doublet_scores_obs_

Another tool to annotate doublets is Doublet Detection. More sophisticated, we use it to actually call doublets.

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata10.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f = doubletdetection.plot.convergence(clf, show=True, p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f2, umap_coords = doubletdetection.plot.umap_plot(adata10.X, doublets, random_state=1, show=True)

In [ ]:
adata10.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
#Thresholding decision: spliced counts
ax=sb.distplot(adata10.obs['n_spliced'], kde=False)
plt.show()

ax=sb.distplot(adata10.obs['n_spliced'][adata10.obs['n_counts']<6000], kde=False, bins=80)
plt.show()

ax=sb.distplot(adata10.obs['n_spliced'][adata10.obs['n_counts']>12000], kde=False, bins=80)
plt.show()

In [ ]:
#Thresholding decision: unspliced counts
ax=sb.distplot(adata10.obs['n_unspliced'], kde=False)
plt.show()

ax=sb.distplot(adata10.obs['n_unspliced'][adata10.obs['n_unspliced']<2000], kde=False, bins=60)
plt.show()

In [ ]:
#Thresholding decision: genes
ax=sb.distplot(adata10.obs['n_genes'], kde=False, bins=60)
plt.show()
ax=sb.distplot(adata10.obs['n_genes'][adata10.obs['n_genes']<1500], kde=False, bins=40)
plt.show()

In [ ]:
print("Before filtering cells:\n", adata10, "\n")

sc.pp.filter_cells(adata10, min_counts=3500)
sc.pp.filter_cells(adata10, max_counts=17000)
adata10 = adata10[adata10.obs['n_unspliced']>500]
sc.pp.filter_cells(adata10, min_genes=800)

print("After filtering cells:\n", adata10)

In [ ]:
sc.pl.violin(adata10, ['n_genes', 'n_counts', 'percent_mito'],
             jitter=0.4, multi_panel=True)
sc.pl.scatter(adata10, 'n_genes', 'percent_mito')

In [ ]:
adata10 = adata10[adata10.obs['percent_mito'] < 0.45, :]
sc.pl.scatter(adata10, 'n_counts', 'n_genes', color='percent_mito', title='inflamed10 (percent_mito)')
print(adata10)

In [ ]:
sc.pl.scatter(adata10, 'n_genes', 'doublet_score', color='doublet', palette=['gray', 'orange'], legend_loc='none', title='Doublet')
adata10 = adata10[adata10.obs['doublet'] != '1']
adata10 = adata10[adata10.obs['doublet_score'] < 0.3]

In [ ]:
sc.pl.highest_expr_genes(adata10, n_top=20)
print(adata10)

## 3. Preprocessing

### 3.1 Normalisation

In [ ]:
adata = adata3.concatenate(adata4, adata6, adata10,
                           batch_key='Sample', 
                           batch_categories=['inflamed3', 'inflamed4', 'inflamed6', 'inflamed10'])
# Apply workaround necessary due to bugs in code
adata.layers['spliced'] = adata.layers['spliced'].astype(float)
adata.layers['unspliced'] = adata.layers['unspliced'].astype(float)

scv.pp.filter_genes(adata, min_cells=5)
scv.pp.filter_genes(adata, min_counts=20)

scv.pp.normalize_per_cell(adata, max_proportion_per_cell=0.05)
scv.pp.log1p(adata)
adata.raw = adata

adata

### 3.2 Cell cycle and proliferation

In [ ]:
ProlifMarkers = [i for i in ProlifMarkers if i in adata.var_names]

sc.tl.score_genes(adata, ProlifMarkers, score_name='Proliferation')

In [ ]:
sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)
adata.obs['CC_difference'] = adata.obs['S_score'] - adata.obs['G2M_score']

### 3.3 Regression and highly variable genes (HVGs) 

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=False)

In [ ]:
corrected = sc.api.pp.mnn_correct(adata[adata.obs['Sample']=='inflamed3'], adata[adata.obs['Sample']=='inflamed4'], 
                                  adata[adata.obs['Sample']=='inflamed6'], adata[adata.obs['Sample']=='inflamed10'],
                                  batch_key='Sample', batch_categories=['inflamed3', 'inflamed4', 'inflamed6', 'inflamed10'],
                                  var_subset=list(adata.var_names[adata.var['highly_variable']]), k=15, var_adj=True, 
                                  do_concatenate=True, save_raw=True, n_jobs=12)

In [ ]:
adata=corrected[0].copy()

In [ ]:
G2M_cells = [name for name in adata.obs_names if adata.obs['G2M_score'][name] > 0]
S_cells = [name for name in adata.obs_names if adata.obs['S_score'][name] > 0]
CC_cells = set(G2M_cells + S_cells)
adata_progen = adata[list(CC_cells),:].copy()

### 3.4 Dimentionality reduction and Principal Component analysis

Next, we can perform PCA...

In [ ]:
sc.pp.regress_out(adata, ['S_score', 'G2M_score'], n_jobs=12)

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=False)

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

...and plot it to look where is the most variance in the dataset coming from.

In [ ]:
# Change plotting settings for more visually pleasing.
scv.settings.set_figure_params('scvelo', dpi=150, vector_friendly=False)

features = ['Sample', 'log_counts', 'phase', 'percent_mito', 'percent_ribo', 'percent_MALAT1']
sc.pl.pca(adata, color=features, ncols=2)

In [ ]:
features = ['LGR5', 'CEACAM1', 'MUC2', 'KRT20', 'CA2', 'BEST4']
sc.pl.pca_overview(adata, color=features, projection='2d', ncols=2)

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)

fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(20,10))

sc.pl.pca(adata, color='Sample', cmap='YlOrRd', alpha=0.5, ax=ax1, show=False, size=15)
adata.uns['pca']['variance_ratio']
ax1.set_aspect(aspect=1)
ax1.set_xlabel('PC1 {}%'.format(round(adata.uns['pca']['variance_ratio'][0]*100, 2)))
ax1.set_ylabel('PC2 {}%'.format(round(adata.uns['pca']['variance_ratio'][1]*100, 2)))
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.pca(adata, color='LGR5', cmap='YlOrRd', ax=ax2, show=False, size=15)
ax2.set_aspect(aspect=1)
ax2.set_xlabel('PC1 {}%'.format(round(adata.uns['pca']['variance_ratio'][0]*100, 2)))
ax2.set_ylabel('PC2 {}%'.format(round(adata.uns['pca']['variance_ratio'][1]*100, 2)))
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.pca(adata, color='MUC2', cmap='YlOrRd', ax=ax3, show=False, size=15)
ax3.set_aspect(aspect=1)
ax3.set_xlabel('PC1 {}%'.format(round(adata.uns['pca']['variance_ratio'][0]*100, 2)))
ax3.set_ylabel('PC2 {}%'.format(round(adata.uns['pca']['variance_ratio'][1]*100, 2)))
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.pca(adata, color='CA1', cmap='YlOrRd', ax=ax4, show=False, size=15)
ax4.set_aspect(aspect=1)
ax4.set_xlabel('PC1 {}%'.format(round(adata.uns['pca']['variance_ratio'][0]*100, 2)))
ax4.set_ylabel('PC2 {}%'.format(round(adata.uns['pca']['variance_ratio'][1]*100, 2)))
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

fig.tight_layout()

fig.savefig('figures/PCA_INFLAMED-supp-fig.pdf', bbox_inches='tight')

## 4. Graph embedding and clustering

### 4.1 Batch correction with bbknn

In [ ]:
sc.external.pp.bbknn(adata, batch_key='Sample', neighbors_within_batch=5, n_pcs=30,
                     approx=False, use_faiss=False, metric='euclidean')

### 4.2 Graph embedding

In [ ]:
sc.tl.umap(adata)

In [ ]:
adata.write("inflamed.h5ad")

In [ ]:
adata = sc.read("inflamed.h5ad")

In [ ]:
sc.set_figure_params(fontsize=15, vector_friendly=False, dpi=150)

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
features = ['LGR5', 'CA1', 'SCGN', 'BEST4', 'LRMP', 'MUC2']

sc.pl.umap(adata, color=features, use_raw=True, ncols=2, save='INFLAMEDMarkers', cmap='YlOrRd')

In [ ]:
sc.pl.umap(adata, color='Proliferation', title="Proliferation score")

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata, color='G2M_score', vmin=-0.25, vmax=0.25, cmap='bwr', title='G2M score', save='InflamedG2M')

In [ ]:
sc.pl.umap(adata, color='S_score', vmin=-0.25, vmax=0.25, cmap='bwr', title='S score', save='InflamedS')

### 4.3 Finding clusters and celltypes 

In [ ]:
scv.settings.set_figure_params(dpi=150, vector_friendly=False)

In [ ]:
sc.tl.leiden(adata, resolution=1)

In [ ]:
sc.pl.umap(adata, color='leiden', legend_loc='on data')

In [ ]:
sc.pl.umap(adata, color='phase', legend_loc='on data')

In [ ]:
sc.tl.leiden(adata, resolution=0.1, restrict_to=('leiden', ['7']))

In [ ]:
adata

In [ ]:
sc.tl.leiden(adata, resolution=0.4, restrict_to=('leiden', ['8']))

In [ ]:
adata.obs['leiden'] = adata.obs['leiden'].replace("4", "3")
adata.obs['leiden'] = adata.obs['leiden'].replace("5", "1")
adata.obs['leiden'] = adata.obs['leiden'].replace("6", "3")
adata.obs['leiden'] = adata.obs['leiden'].replace("7,0", "4")
adata.obs['leiden'] = adata.obs['leiden'].replace("7,1", "5")
adata.obs['leiden'] = adata.obs['leiden'].replace("8,0", "6")
adata.obs['leiden'] = adata.obs['leiden'].replace("8,1", "7")
adata.obs['leiden'] = adata.obs['leiden'].replace("8,2", "8")
adata.obs['leiden'] = adata.obs['leiden'].replace("9", "2")
adata.obs['leiden'] = adata.obs['leiden'].replace("10", "9")
adata.obs['leiden'] = adata.obs['leiden'].replace("11", "10")
adata.obs['leiden'] = adata.obs['leiden'].replace("12", "11")
adata.obs['leiden'] = adata.obs['leiden'].replace("13", "12")

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata, color='leiden', legend_loc='on data', save='Inflamed_leiden', title='')

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon')
sc.tl.filter_rank_genes_groups(adata, min_in_group_fraction=0.5, 
                               max_out_group_fraction=0.5, min_fold_change=1.5)
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5, key='rank_genes_groups_filtered', 
                                standard_scale='var', color_map='bwr', dendrogram=False)

In [ ]:
adata.obs['Celltype'] = adata.obs['leiden']
adata.obs['Celltype'] = adata.obs['Celltype'].replace("0", "Goblet")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("1", "TA Goblet")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("2", "Stem")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("3", "TA Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("4", "TA Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("5", "TA Goblet")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("6", "TA Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("7,0", "Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("7,1", "CT Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("8,0", "TA BEST4+")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("8,1", "BEST4+")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("8,2", "CT BEST4+")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("9", "Stem")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("10", "EECs")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("11", "TA SOX4+")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("12", "Tuft")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("13", "CT Goblet")

adata.obs['Celltype'] = adata.obs['Celltype'].astype('category')
Celltype_order = ['Stem',
                  'TA SOX4+',
                  'TA Colonocytes', 'Colonocytes', 'CT Colonocytes',
                  'TA BEST4+', 'BEST4+', 'CT BEST4+', 
                  'TA Goblet', 'Goblet', 'CT Goblet',                  
                  'EECs' ,'Tuft']
adata.obs['Celltype'].cat.reorder_categories(Celltype_order, inplace=True)

In [ ]:
vega_colors = np.array(sc.pl.palettes.vega_20_scanpy)

celltype_colors = np.zeros(len(set(adata.obs['Celltype'])))
celltype_colors = celltype_colors.astype('U7')

celltype_colors[[0]] =  vega_colors[[1]] # Stem color / orange
celltype_colors[[1]] = '#FFCC00' # TA SOX4+ colors / yellow
celltype_colors[[2, 3, 4]] = vega_colors[[12, 10, 15]]  # Colono colors / reds
celltype_colors[[5, 6, 7]] = vega_colors[[0, 8, 17]]  # BEST4 colors / blues
celltype_colors[[8, 9, 10]] = vega_colors[[2, 7, 11]]  # Goblet colors / greens
celltype_colors[[11]] = '#E53333'  # EECs / red
celltype_colors[[12]] = '#A9A9A9'  # Tuft / grey

adata.uns['Celltype_colors'] = celltype_colors

In [ ]:
sc.set_figure_params(fontsize=10, vector_friendly=False, dpi=150)
#scv.settings.set_figure_params(dpi=150, vector_friendly=False)

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata, color='Celltype', legend_loc='right margin', alpha=0.7, size=20, save='InflamedCelltype', title='')

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon')
sc.tl.filter_rank_genes_groups(adata, min_in_group_fraction=0.25, 
                               max_out_group_fraction=0.75, min_fold_change=1.5)
RevCelltype_order = Celltype_order
RevCelltype_order.reverse()
adata.obs['Celltype'].cat.reorder_categories(RevCelltype_order, inplace=True)
sc.pl.rank_genes_groups_dotplot(adata, n_genes=4, key='rank_genes_groups_filtered', groupby='Celltype',
                                standard_scale='var', color_map='bwr', dendrogram=False, save='DE_inflamed',
                                figsize=(10,3))

In [ ]:
adata.obs['Celltype'].cat.reorder_categories(Celltype_order, inplace=True)

## Cell cycle comparison

In [ ]:
adata.obs['Condition'] = 'INFLAMED'
sample_celltype_sizes = adata.obs.groupby(['Sample'])['Celltype'].value_counts()
phase_summary = pd.DataFrame(adata.obs.groupby(['Condition', 'Sample', 'Celltype'])['phase'].value_counts()).rename(columns={'phase': "Counts"})

In [ ]:
phase_summary = phase_summary.reset_index()

phase_summary['Percentage'] = [phase_summary.loc[index, 'Counts']*100/sample_celltype_sizes[phase_summary.loc[index, 'Sample'], phase_summary.loc[index, 'Celltype']] 
                                  for index in phase_summary.index]

phase_summary.to_csv('inflamed_phase_summary.csv')

## 5. Trajectory analysis

### 5.1 PAGA

In [ ]:
adata.obs['SubCelltype'] = adata.obs['Celltype']
for celltype in adata.obs['Celltype'].cat.categories.tolist():
    sc.tl.leiden(adata, resolution=0.6, restrict_to=('SubCelltype', [celltype]), key_added='SubCelltype')

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata, color='SubCelltype', legend_loc='none', save='Inflamed_subcelltype', title='', legend_fontweight='light')

In [ ]:
sc.tl.paga(adata, groups='SubCelltype')

In [ ]:
sc.pl.paga(adata, color='Celltype', threshold=0.7, 
           node_size_scale=0.6, edge_width_scale=0.1, node_size_power=0.2, 
           layout='fr', text_kwds={'alpha':0}, frameon=False, save='Inflamed')

In [ ]:
sc.pl.paga(adata, color='SOX4', threshold=0.7, 
           node_size_scale=0.8, edge_width_scale=0.2, node_size_power=0.3, 
           layout='fr', text_kwds={'alpha':0})

In [ ]:
sc.pl.paga(adata, color='CD74', threshold=0.7, 
           node_size_scale=0.8, edge_width_scale=0.2, node_size_power=0.3, 
           layout='fr', text_kwds={'alpha':0}, title='CD74')

In [ ]:
sc.pl.paga(adata, color=['LGR5', 'KRT20'], threshold=0.7, 
           node_size_scale=0.8, edge_width_scale=0.2, node_size_power=0.3, 
           layout='fr', text_kwds={'alpha':0})

### 5.2 Graph embedding based on PAGA

In [ ]:
sc.tl.draw_graph(adata, init_pos='paga')

In [ ]:
sc.pl.draw_graph(adata, color='Celltype', title='',
                 size=10, alpha=0.7, legend_fontsize=6, frameon=False, edges=False, legend_loc='none', save='test')

In [ ]:
#sc.set_figure_params(fontsize=15, vector_friendly=False, dpi=150)
scv.settings.set_figure_params(dpi=150, vector_friendly=False)

In [ ]:
sc.pl.draw_graph(adata, color='Proliferation', legend_fontsize=12, frameon=False, edges=False, title="Proliferation score",
                 size=30, alpha=0.75, save='INFLAMEDprolif')

In [ ]:
sc.pl.draw_graph(adata, color='KRT20', legend_fontsize=12, frameon=False, edges=False,
                 size=30, alpha=0.75, save='INFLAMEDKRT20')

In [ ]:
sc.pl.draw_graph(adata, color=['KRT20', 'SCGN', 'LRMP'], legend_fontsize=12, frameon=False, edges=False,
                 size=20, alpha=0.9)

### 5.3 Diffusion-based pseudotime

In [ ]:
# Set one of the stem cells as the starting point for the pseudotime.
# Biased - compare with latent time. 
adata.uns['iroot'] = np.flatnonzero(adata.obs['Celltype']  == 'Stem')[0]

sc.tl.diffmap(adata)
sc.tl.dpt(adata)

In [ ]:
sc.pl.draw_graph(adata, color='dpt_pseudotime', legend_loc='on data', size=20, legend_fontsize=8, frameon=False, edges=False, cmap='viridis')

## XX.1 Fraction of cells statistics

In [ ]:
celltype_sizes = adata.obs['Celltype'].value_counts()

sample_summary = pd.DataFrame(adata.obs.groupby(['Celltype'])['Sample'].value_counts()).rename(columns={'Sample': "Counts"})

In [ ]:
sample_summary = sample_summary.reset_index()

sample_summary['Fraction'] = [sample_summary.loc[index, 'Counts']/celltype_sizes[sample_summary.loc[index, 'Celltype']] 
                                  for index in sample_summary.index]
del sample_summary['Counts']

sample_summary.to_csv('inflamed_sample_summary.csv')

## 6. Velocity

### 6.1 Velocity computation 

In [ ]:
scv.pp.moments(adata, n_pcs=30, n_neighbors=20)
scv.tl.recover_dynamics(adata)

In [ ]:
scv.tl.velocity(adata, mode='dynamical', groupby='Celltype')
scv.tl.velocity_graph(adata)
scv.tl.recover_latent_time(adata)

In [ ]:
adata.write("inflamed_velo.h5ad")

In [ ]:
adata = sc.read("inflamed_velo.h5ad")

### 6.3 Plotting velocity on embeddings

In [ ]:
#sc.set_figure_params(fontsize=15, vector_friendly=False, dpi=150)
scv.settings.set_figure_params(dpi=150, vector_friendly=False, figsize=[5,4])

In [ ]:
scv.pl.velocity_embedding_grid(adata, basis='umap', 
                                 legend_fontsize=10, title='', 
                                 smooth=.52, min_mass=0, color='Celltype',
                                 alpha=0.7, size=30, fontsize=30, legend_loc='none', save="Inflamed_grid")

In [ ]:
scv.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300, figsize=[4,4], format='png')
scv.pl.velocity_embedding_stream(adata, basis='umap', color='Celltype',
                               legend_fontsize=10, smooth=.7, min_mass=0.1, 
                               size=40, alpha=0.9, legend_loc='none', save='Inflamed', title='')

In [ ]:
scv.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300, figsize=[4,4], format='png')
gene = 'KRT20'
scv.pl.velocity_embedding_stream(adata, basis='umap', color='{}'.format(gene), cmap='YlOrRd',
                               legend_fontsize=10, smooth=.7, min_mass=0.1, 
                               size=40, alpha=0.9, legend_loc='none', title='{} expression'.format(gene), save='{}INFLAMED'.format(gene))

In [ ]:
scv.pl.velocity_graph(adata, basis='umap', color='Celltype', legend_loc='right margin')

### 6.4 Identification of marker genes driving the velocity.

In [ ]:
scv.tl.rank_velocity_genes(adata, groupby='Celltype')

pd.DataFrame(adata.uns['rank_velocity_genes']['names']).head(n=10)

In [ ]:
scv.pl.velocity(adata, var_names=['MPZL2'], colorbar=True, ncols=1, 
                layers=['velocity', 'Ms'], use_raw=False, cmap='RdBu_r', color_map='RdBu_r', 
                size=20, alpha=0.7)

In [ ]:
scv.pl.velocity(adata, var_names=['LGR5', 'FN1', 'PTPRO', 'ZBTB38', 'FAM162A', 'PON2', 'LMAN1', 'KRT20', 'CA1', 'TFF3'], colorbar=True, ncols=1, 
                layers=['velocity', 'Ms', 'Mu'], use_raw=False, cmap='RdBu_r', color_map='RdBu_r', 
                size=20, alpha=0.7)

In [ ]:
scv.pl.scatter(adata, basis=['PTPRO', 'ZBTB38', 'FAM162A', 'PON2'], fontsize=16, size=100, linewidth=3,
               frameon=False, legend_loc='none', color='Celltype')

In [ ]:
scv.pl.scatter(adata, basis='PTPRO', fontsize=16, size=100, linewidth=3,
               frameon=False, legend_loc='none', color='PTPRO', layer='velocity', cmap='RdBu_r')

In [ ]:
scv.pl.scatter(adata, x='latent_time', y=['PTPRO', 'ZBTB38', 'FAM162A', 'PON2'], fontsize=16, size=100,
               n_convolve=None, frameon=False, legend_loc='none', color='Celltype', layer='Ms')

In [ ]:
top_genes = adata.var_names[adata.var.fit_likelihood.argsort()[::-1]][:300]
scv.pl.heatmap(adata, var_names=top_genes, tkey='latent_time', n_convolve=100, col_color='Celltype', color_map='RdBu_r', xkey='Ms')

### 6.5 Latent time

In [ ]:
scv.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300, figsize=[4,4], format='png')
scv.pl.scatter(adata, color='latent_time', fontsize=24, size=100, basis='umap',
               color_map='gnuplot', perc=[2, 98], colorbar=True, rescale_color=[0,1], title='', save='Inflamed_time')

In [ ]:
scv.tl.velocity_confidence(adata)

In [ ]:
scv.pl.scatter(adata, color='velocity_confidence', fontsize=24, size=100, colorbar=True, rescale_color=[0,1], color_map='RdBu_r')
scv.pl.scatter(adata, color='velocity_confidence_transition', fontsize=24, size=100, colorbar=True, rescale_color=[0,1], color_map='RdBu_r')

### 6.6 PAGA with velocity

In [ ]:
sc.tl.paga(adata, use_rna_velocity=False, groups='leiden')
sc.tl.paga(adata, use_rna_velocity=True, groups='leiden')

In [ ]:
sc.pl.paga(adata, color='Celltype', threshold=0.1, fontsize=5,
           node_size_scale=1, edge_width_scale=0.5, node_size_power=0.5, 
           layout='fr', transitions='transitions_confidence', arrowsize=10)

## RAUL

In [ ]:
Rgenes = ['Celltype','ATF3', 'ATF4', 'HOXB13']

sc.pl.umap(adata, color=Rgenes, use_raw=True, ncols=2, cmap='RdBu_r', save='RaulColonInflamed', legend_loc='right margin', size=60, alpha=0.7)

In [ ]:
sc.pl.dotplot(adata, Rgenes[1:], groupby='Celltype', var_group_rotation=0, dendrogram=False, color_map='bwr', standard_scale='var', save='RaulColonInflamedDotplot', use_raw=True)

# LOAD DATA

In [ ]:
adata = sc.read("inflamed_velo.h5ad")

# IBD genes

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)

In [ ]:
with open('IBD_figures/UC.txt', 'r') as f:
    UC_genes = f.read().splitlines()
    
with open('IBD_figures/CD.txt', 'r') as f:
    CD_genes = f.read().splitlines()
    
with open('IBD_figures/IBD.txt', 'r') as f:
    IBD_genes = f.read().splitlines()

In [ ]:
UC_genes = list(set(UC_genes))
print(len(UC_genes))
UC_genes = [feature for feature in UC_genes if feature in adata.var_names]
print(len(UC_genes))

CD_genes = list(set(CD_genes))
print(len(CD_genes))
CD_genes = [feature for feature in CD_genes if feature in adata.var_names]
print(len(CD_genes))

IBD_genes = list(set(IBD_genes))
print(len(IBD_genes))
IBD_genes = [feature for feature in IBD_genes if feature in adata.var_names]
print(len(IBD_genes))

In [ ]:
sc.tl.score_genes(adata, UC_genes, score_name='UC_genes')

In [ ]:
sc.tl.score_genes(adata, CD_genes, score_name='CD_genes')

In [ ]:
sc.tl.score_genes(adata, IBD_genes, score_name='IBD_genes')

In [ ]:
sc.pl.umap(adata, color='IBD_genes', cmap='bwr')

In [ ]:
sc.pl.umap(adata, color='CD74', cmap='bwr')

In [ ]:
adata

# 9. Analysis of subset of EECs and Tuft

## 9.1 EECs

In [ ]:
adata_EECs.obs['Sample']

In [ ]:
EECs_names = adata1.obs_names[adata1.obs['Celltype']=='EECs']

In [ ]:
EECs_names_short = [i[:-10] for i in list(EECs_names.values)]

In [ ]:
adata_EECs = adata[adata.obs_names.isin(EECs_names_short)].copy()

In [ ]:
sc.pp.highly_variable_genes(adata_EECs, n_top_genes=2000, subset=False)

In [ ]:
corrected = sc.api.pp.mnn_correct(adata_EECs[adata_EECs.obs['Sample']=='inflamed3'], adata_EECs[adata_EECs.obs['Sample']=='inflamed4'], 
                                  adata_EECs[adata_EECs.obs['Sample']=='inflamed6'],
                                  batch_key='Sample', batch_categories=['inflamed3', 'inflamed4', 'inflamed6'],
                                  var_subset=list(adata_EECs.var_names[adata_EECs.var['highly_variable']]), k=15, var_adj=True, 
                                  do_concatenate=True, save_raw=True, n_jobs=12)

In [ ]:
adata_EECs = corrected[0].copy()

In [ ]:
sc.pp.highly_variable_genes(adata_EECs, n_top_genes=2000, subset=False)

In [ ]:
sc.tl.pca(adata_EECs, svd_solver='arpack')

In [ ]:
sc.external.pp.bbknn(adata_EECs, batch_key='Sample', neighbors_within_batch=5, n_pcs=30,
                     approx=False, use_faiss=False, metric='euclidean')

In [ ]:
sc.tl.umap(adata_EECs)

In [ ]:
sc.pl.umap(adata_EECs, color='Sample', size=800, save='_ULCERATED_EECs_sample')

In [ ]:
sc.pl.umap(adata_EECs, color=['PYY', 'SCGN'], cmap='YlOrRd', size=800, save='_ULCERATED_EECs_markers')

In [ ]:
sc.tl.leiden(adata_EECs, resolution=0.5)

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata_EECs, color='leiden', legend_loc='right margin', title='', size=800, save='_ULCERATED_EECs_cluster')

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.tl.rank_genes_groups(adata_EECs, groupby='leiden', method='wilcoxon')
sc.tl.filter_rank_genes_groups(adata_EECs, min_in_group_fraction=0.6, 
                               max_out_group_fraction=0.25, min_fold_change=1.5)
sc.pl.rank_genes_groups_dotplot(adata_EECs, n_genes=5, key='rank_genes_groups_filtered', 
                                standard_scale='var', color_map='bwr', dendrogram=False, save='_ULCERATED_EECs_DEGs')

In [ ]:
adata_EECs

In [ ]:
sc.pl.umap(adata_EECs, size=800, color=['CDKN1C', 'STMN1', 'HES6', 'SPINK4', 'ANXA13'], cmap='YlOrRd', ncols=2, save='ULCERATED_EECs_GCG-_PYY-_markers')

In [ ]:
sc.pl.umap(adata_EECs, color=['NUCB2', 'SST', 'GCG', 'PYY', 'TIMP1', 'STMN1', 'PTMA'], 
           cmap='YlOrRd', size=800, ncols=2, use_raw=True, save='_ULCERATED_EECs_hormones')

In [ ]:
sc.pl.umap(adata_EECs, color=['PYY', 'GCG', 'SST', 'TPH1', 'SCT', 'SCGN'], 
           cmap='YlOrRd', size=800, ncols=2, use_raw=True, save='_ULCERATED_EECs_NewMarkers')

In [ ]:
sc.pl.umap(adata_EECs, color=['SST', 'GCG', 'TIMP1', 'PYY'], 
           cmap='YlOrRd', size=800, ncols=2, use_raw=True, save='_ULCERATED_EECs_hormones_selected')

In [ ]:
scv.pp.moments(adata_EECs, n_pcs=30, n_neighbors=20)
scv.tl.recover_dynamics(adata_EECs)

In [ ]:
scv.tl.velocity(adata_EECs, mode='dynamical', groupby='leiden')
scv.tl.velocity_graph(adata_EECs)#, n_neighbors=12)
scv.tl.recover_latent_time(adata_EECs)

In [ ]:
scv.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300, figsize=[4,4], format='png')

scv.pl.velocity_embedding_stream(adata_EECs, basis='umap', cmap='Reds',
                                 legend_fontsize=8, title='', 
                                 smooth=.7, min_mass=0.1, color='leiden',
                                 alpha=0.9, fontsize=30, legend_loc='none', size=800, save='_ULCERATED_EECs_velocity')

In [ ]:
adata_EECs.write("ulcerated_EECs_velo.h5ad")

## 9.1 Tuft

In [ ]:
Tuft_names = adata1.obs_names[adata1.obs['Celltype']=='Tuft']

In [ ]:
Tuft_names_short = [i[:-10] for i in list(Tuft_names.values)]

In [ ]:
adata_Tuft = adata[adata.obs_names.isin(Tuft_names_short)].copy()

In [ ]:
sc.pp.highly_variable_genes(adata_Tuft, n_top_genes=2000, subset=False)

In [ ]:
corrected = sc.api.pp.mnn_correct(adata_Tuft[adata_Tuft.obs['Sample']=='inflamed3'], adata_Tuft[adata_Tuft.obs['Sample']=='inflamed4'], 
                                  adata_Tuft[adata_Tuft.obs['Sample']=='inflamed6'],
                                  batch_key='Sample', batch_categories=['inflamed3', 'inflamed4', 'inflamed6'],
                                  var_subset=list(adata_Tuft.var_names[adata_Tuft.var['highly_variable']]), k=15, var_adj=True, 
                                  do_concatenate=True, save_raw=True, n_jobs=12)

In [ ]:
adata_Tuft = corrected[0].copy()

In [ ]:
sc.pp.highly_variable_genes(adata_Tuft, n_top_genes=2000, subset=False)

In [ ]:
sc.tl.pca(adata_Tuft, svd_solver='arpack')

In [ ]:
sc.external.pp.bbknn(adata_Tuft, batch_key='Sample', neighbors_within_batch=5, n_pcs=30,
                     approx=False, use_faiss=False, metric='euclidean')

In [ ]:
sc.tl.umap(adata_Tuft)

In [ ]:
sc.pl.umap(adata_Tuft, color='Sample', size=600, save='_ULCERATED_Tuft_sample')

In [ ]:
sc.pl.umap(adata_Tuft, color=['LRMP', 'SOX4'], cmap='YlOrRd', size=600, save='_ULCERATED_Tuft_markers')

In [ ]:
sc.tl.leiden(adata_Tuft, resolution=0.6)

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.umap(adata_Tuft, color='leiden', legend_loc='right margin', title='', size=600, save='_ULCERATED_Tuft_clusters')

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.tl.rank_genes_groups(adata_Tuft, groupby='leiden', method='wilcoxon')
sc.tl.filter_rank_genes_groups(adata_Tuft, min_in_group_fraction=0.6, 
                               max_out_group_fraction=0.25, min_fold_change=1.5)
sc.pl.rank_genes_groups_dotplot(adata_Tuft, n_genes=5, key='rank_genes_groups_filtered', 
                                standard_scale='var', color_map='bwr', dendrogram=False, save='_ULCERATED_Tuft_DEGs')

In [ ]:
scv.pp.moments(adata_Tuft, n_pcs=30, n_neighbors=20)
scv.tl.recover_dynamics(adata_Tuft)

In [ ]:
scv.tl.velocity(adata_Tuft, mode='dynamical', groupby='leiden')
scv.tl.velocity_graph(adata_Tuft)#, n_neighbors=12)
scv.tl.recover_latent_time(adata_Tuft)

In [ ]:
scv.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300, figsize=[4,4], format='png')

scv.pl.velocity_embedding_stream(adata_Tuft, basis='umap', cmap='Reds',
                                 legend_fontsize=8, title='', 
                                 smooth=.7, min_mass=0.1, color='leiden',
                                 alpha=0.9, fontsize=30, legend_loc='none', size=600, save='_ULCERATED_Tuft_velocity')

In [ ]:
adata_Tuft.write("ulcerated_Tuft_velo.h5ad")

In [ ]:
adata.obs['CD74_expression'] = adata.raw[:,'CD74'].X.todense()

In [ ]:
len(adata.obs_names[adata.obs['CD74_expression'] > 0.3970])

In [ ]:
len(adata.obs_names[adata.obs['CD74_expression'] <= 0.3970])

In [ ]:
len(adata)

In [ ]:
3937*100 / 4354

In [ ]:
417*100 / 4354

# PCA 

In [ ]:
sc.pl.pca_variance_ratio(adata, log=False)

In [ ]:
sc.pl.pca_loadings(adata, components = '1,2,3,4,5,6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20', include_lowest=False)

In [ ]:
adata.obs['PC1'] = adata.obsm['X_pca'][:, 0]
adata.obs['PC2'] = adata.obsm['X_pca'][:, 1]
adata.obs['PC3'] = adata.obsm['X_pca'][:, 2]
adata.obs['PC4'] = adata.obsm['X_pca'][:, 3]
adata.obs['PC5'] = adata.obsm['X_pca'][:, 4]
adata.obs['PC6'] = adata.obsm['X_pca'][:, 5]
adata.obs['PC7'] = adata.obsm['X_pca'][:, 6]
adata.obs['PC8'] = adata.obsm['X_pca'][:, 7]
adata.obs['PC9'] = adata.obsm['X_pca'][:, 8]
adata.obs['PC10'] = adata.obsm['X_pca'][:, 9]
adata.obs['PC11'] = adata.obsm['X_pca'][:, 10]
adata.obs['PC12'] = adata.obsm['X_pca'][:, 11]
adata.obs['PC13'] = adata.obsm['X_pca'][:, 12]
adata.obs['PC14'] = adata.obsm['X_pca'][:, 13]
adata.obs['PC15'] = adata.obsm['X_pca'][:, 14]
adata.obs['PC16'] = adata.obsm['X_pca'][:, 15]
adata.obs['PC17'] = adata.obsm['X_pca'][:, 16]
adata.obs['PC18'] = adata.obsm['X_pca'][:, 17]
adata.obs['PC19'] = adata.obsm['X_pca'][:, 18]
adata.obs['PC20'] = adata.obsm['X_pca'][:, 19]

In [ ]:
pcs = ['PC' + str(i) for i in range(1,21)]

sc.pl.umap(adata, color=pcs, cmap='bwr', ncols=3)

In [ ]:
sc.pl.umap(adata, color='HEPACAM2', cmap='YlOrRd')